# NLP Assignment 5: Token Classification (POS Tagging)
**Internship Task 5: February 2026**
**Submitted by:** Shankar Reddy | **Intern ID:** [IN226102102]

### Objective:
Fine-tune a DistilBERT model for Part-of-Speech (POS) tagging using the CoNLL-2003 dataset.
- **Task:** Sequence Labeling (Token Classification)
- **Model:** distilbert-base-uncased
- **Metric:** seqeval (F1, Precision, Recall)

In [6]:
!pip install evaluate seqeval accelerate

  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)

   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ---------------------------------------- 0/2 [accelerate]
   ------------------

In [15]:
!pip install -U accelerate transformers[torch]

  Using cached transformers-5.5.0-py3-none-any.whl.metadata (32 kB)
Using cached transformers-5.5.0-py3-none-any.whl (10.2 MB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.4.0
    Uninstalling transformers-5.4.0:
      Successfully uninstalled transformers-5.4.0


In [3]:
# Install missing components
!pip install transformers datasets evaluate seqeval accelerate -q

import torch, warnings, random
import numpy as np
import pandas as pd
from datasets import load_dataset
import evaluate  # Now this will work!
from transformers import (AutoTokenizer, AutoModelForTokenClassification, 
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)

warnings.filterwarnings("ignore")
SEED, MODEL_NAME = 42, "distilbert-base-uncased"
TRAIN_SIZE, EPOCHS, LR = 500, 1, 2e-5

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Environment Ready. Using: {device}")

🚀 Environment Ready. Using: cpu


In [4]:
# Task 1: Dataset Selection - Universal Dependencies (EWT)
# Using the 'commul' path which is pre-formatted for modern datasets library
raw_datasets = load_dataset("commul/universal_dependencies", "en_ewt")

# Select small subsets for the demo
train_dataset = raw_datasets["train"].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_dataset = raw_datasets["dev"].select(range(150)) # Note: some UD configs use 'dev' instead of 'validation'

# Map labels
label_list = raw_datasets["train"].features["upos"].feature.names
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

print(f"✅ Dataset Loaded Successfully!")
print(f"✅ Total POS Labels: {len(label_list)}")
print(f"✅ Subset Sizes - Train: {len(train_dataset)}, Val: {len(val_dataset)}")

✅ Dataset Loaded Successfully!
✅ Total POS Labels: 18
✅ Subset Sizes - Train: 500, Val: 150


In [5]:
# Re-defining the missing constants
MAX_LENGTH = 128
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
EPOCHS = 1
MODEL_NAME = "distilbert-base-uncased"

print("✅ Constants defined. You can now run the mapping cell.")

✅ Constants defined. You can now run the mapping cell.


In [6]:
# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def align_labels_func(examples):
    # Tokenize the words. 'is_split_into_words=True' is CRITICAL for POS tagging
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        is_split_into_words=True, 
        truncation=True, 
        max_length=MAX_LENGTH
    )

    labels = []
    # Loop through the UPOS tags in the batch
    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens like [CLS] and [SEP] get -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # This is the first subword of a new word
                label_ids.append(label[word_idx])
            else:
                # This is a subsequent subword (##ker), so we ignore it
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply the mapping to our subsets
tokenized_train = train_dataset.map(align_labels_func, batched=True)
tokenized_val = val_dataset.map(align_labels_func, batched=True)

print("✅ Tokenization and Alignment Complete.")

✅ Tokenization and Alignment Complete.


In [7]:
# Task 3: Load Model with correct label count
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
).to(device)

# Task 4: Define Training Arguments
training_args = TrainingArguments(
    output_dir="./distilbert_pos_results",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    report_to="none" # Prevents wandb/tensorboard popups
)

# Use DataCollator for dynamic padding of sequence labels
data_collator = DataCollatorForTokenClassification(tokenizer)

# Updated Trainer Initialization
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer  # <--- Changed from 'tokenizer' to 'processing_class'
)

print("🚀 Argument fixed. Starting Training...")
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


🚀 Argument fixed. Starting Training...


Step,Training Loss
10,2.802473
20,2.532178
30,2.370021


TrainOutput(global_step=32, training_loss=2.552693337202072, metrics={'train_runtime': 80.6815, 'train_samples_per_second': 6.197, 'train_steps_per_second': 0.397, 'total_flos': 6043433006160.0, 'train_loss': 2.552693337202072, 'epoch': 1.0})

In [8]:
from transformers import Trainer, TrainingArguments, DataCollatorForTokenClassification

# Initialize the Trainer object
training_args = TrainingArguments(
    output_dir="./pos_results",
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    disable_tqdm=True, # This prevents the visual glitch
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer
)
print("✅ Trainer 'trainer' is now defined and ready!")

✅ Trainer 'trainer' is now defined and ready!


In [10]:
import evaluate
import numpy as np

# Load the seqeval metric specifically for Token Classification
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (-100) and convert IDs to actual Labels
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# Update the trainer with this new function
trainer.compute_metrics = compute_metrics
print("✅ Metrics function updated. Ready to evaluate again!")

✅ Metrics function updated. Ready to evaluate again!


In [11]:
# Task 4 & 5: Training and Evaluation (Bypass Version)
print("🚀 Training and Evaluating...")

# 1. Temporarily disable the notebook progress bar to avoid the RuntimeError
trainer.args.disable_tqdm = True 
trainer.args.report_to = [] # Disable external logging

# 2. Re-run train and evaluate in the same call
trainer.train()
metrics = trainer.evaluate()

# 3. Create the Results Table
import pandas as pd
eval_results = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1 Score", "Accuracy"],
    "Value": [
        round(metrics.get("eval_precision", 0), 4),
        round(metrics.get("eval_recall", 0), 4),
        round(metrics.get("eval_f1", 0), 4),
        round(metrics.get("eval_accuracy", 0), 4)
    ]
})

print("✅ Success! Here are your Task 5 Metrics:")
display(eval_results)

🚀 Training and Evaluating...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '90.53', 'train_samples_per_second': '5.523', 'train_steps_per_second': '0.353', 'train_loss': '1.231', 'epoch': '1'}
{'eval_loss': '0.958', 'eval_precision': '0.7528', 'eval_recall': '0.7176', 'eval_f1': '0.7347', 'eval_accuracy': '0.7774', 'eval_runtime': '7.868', 'eval_samples_per_second': '19.07', 'eval_steps_per_second': '2.415', 'epoch': '1'}
✅ Success! Here are your Task 5 Metrics:


,Metric,Value
0,Precision,0.7528
1,Recall,0.7176
2,F1 Score,0.7347
3,Accuracy,0.7774


In [12]:
def get_predictions(sentence):
    model.eval()
    words = sentence.split()
    
    # Tokenize the custom sentence
    inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt").to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    
    # Get IDs and filter out special tokens ([CLS], [SEP])
    preds = torch.argmax(logits, dim=2).squeeze().tolist()
    
    word_ids = inputs.word_ids()
    final_labels = []
    prev_word_idx = None
    
    for i, word_idx in enumerate(word_ids):
        if word_idx is not None and word_idx != prev_word_idx:
            final_labels.append(id2label[preds[i]])
        prev_word_idx = word_idx
        
    return pd.DataFrame({"Word": words, "Predicted POS": final_labels})

# Required example for Task 6
test_sentence = "John works at Google in California"
print(f"🔮 Prediction Proof for Assignment 5:")
display(get_predictions(test_sentence))

🔮 Prediction Proof for Assignment 5:


,Word,Predicted POS
0,John,PROPN
1,works,VERB
2,at,ADP
3,Google,NOUN
4,in,ADP
5,California,NOUN


In [16]:
# Task 7 & 8: Technical Report
report = """
### Task 7: POS vs Chunking Comparison ###

| Feature      | POS Tagging          | Chunking                |
|--------------|----------------------|-------------------------|
| Level        | Word-level (Atomic)  | Phrase-level (Groups)   |
| Difficulty   | Easier               | Medium                  |
| Example      | 'John' -> PROPN      | 'The Texas Lab' -> [NP] |

### Task 8: Technical Insights ###
- Challenge: Subword Alignment using -100 masking.
- Observation: DistilBERT is 60% faster than standard BERT.
- Result: Final Accuracy achieved was 77.74%.
"""

print(report)


### Task 7: POS vs Chunking Comparison ###

| Feature      | POS Tagging          | Chunking                |
|--------------|----------------------|-------------------------|
| Level        | Word-level (Atomic)  | Phrase-level (Groups)   |
| Difficulty   | Easier               | Medium                  |
| Example      | 'John' -> PROPN      | 'The Texas Lab' -> [NP] |

### Task 8: Technical Insights ###
- Challenge: Subword Alignment using -100 masking.
- Observation: DistilBERT is 60% faster than standard BERT.
- Result: Final Accuracy achieved was 77.74%.

